# Introduction to linear regression

The movie [Moneyball](https://en.wikipedia.org/wiki/Moneyball_(film)) focuses on the "quest for the secret of success in baseball." It follows a low-budget team, the Oakland Athletics, who believed that underused statistics, such as a player's ability to get on base, better predict the ability to score runs than typical statistics like home runs, RBIs (runs batted in), and batting average. Obtaining players who excelled in these underused statistics turned out to be much more affordable for the team.

In this lab we'll be looking at data from all 30 Major League Baseball teams and examining the linear relationship between runs scored in a season and a number of other player statistics. Our aim will be to summarize these relationships both graphically and numerically in order to find which variable, if any, helps us best predict a team's runs scored in a season.

## The data

Let's load up the data for the 2011 season.

In [ ]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import io
import requests
from plotnine import *

df_url = 'https://raw.githubusercontent.com/akmand/datasets/master/openintro/mlb11.csv'
url_content = requests.get(df_url, verify=False).content
mlb11 = pd.read_csv(io.StringIO(url_content.decode('utf-8')))
mlb11.head()

In addition to runs scored, there are seven traditionally used variables in the data set, namely
- `at-bats`,
- `hits`,
- home runs (`homeruns`),
- batting average (`bat_avg`),
- `strikeouts`,
- stolen bases (`stolen_bases`), and
- `wins`.

There are also three newer variables
- on-base percentage (`new_onbase`),
- slugging percentage (`new_slug`), and
- on-base plus slugging (`new_obs`).


For the first portion of the analysis we'll consider the seven traditional variables. After that, you'll work with the newer variables.

### Exercise 1

- What type of plot would you use to display the relationship between <code>runs</code> and one of the other numerical variables?
- Use one of these plots to visualize the relationship between `at_bats` and `runs` (using `at_bats` as the predictor variable).
- If you knew a team's <code>at_bats</code>, would you be comfortable using a linear model to predict the number of runs?


In [ ]:
# Plot the relationship between `at_bats` and `runs` (using `at_bats` as predictor).



If the relationship looks linear, we can quantify the strength of the relationship with the correlation coefficient.

In [ ]:
mlb11['runs'].corr(mlb11['at_bats'])

## Sum of squared residuals

Think back to the way that we described the distribution of a single variable. Recall that we discussed characteristics such as **center**, **spread**, and **shape**. It's also useful to be able to describe the relationship of two numerical variables, such as `runs` and `at_bats` above.

### Exercise 2

- Looking at your plot from the previous exercise, describe the relationship between these two variables.
- Make sure to discuss the **form**, **direction**, and **strength** of the relationship as well as any **unusual observations**.

Recall that the difference between the observed values and the values predicted by the line are called **residuals**. Note that the data set has 30 observations in total, hence there are 30 residuals.

To visualize the residuals of a linear regression, we can use `residplot()` function from `seaborn`.

In [ ]:
# Import the required packages and set figure parameters

import seaborn as sns
import matplotlib.pyplot as plt
%matplotlib inline
%config InlineBackend.figure_format = 'retina'
plt.style.use('ggplot')
plt.rcParams['figure.figsize'] = (10,5)

# Make the residue plot

sns.residplot(x='at_bats', y='runs', data=mlb11, color='red')
plt.show();

## The linear model

In order to determine the best fit line we can use `statsmodel`, a very useful module for the estimation of many different statistical models, as well as for conducting statistical tests, and statistical data exploration.

In [ ]:
# Import the `statsmodels` library
import statsmodels.api as sm

# Set the formula string (which variables to compare, "reponse ~ predictor")
# and the dataframe to use.

formula_string = "runs ~ at_bats"
dataframe = mlb11

# Fit the linear model

model = sm.formula.ols(formula = formula_string, data = dataframe)
model_fitted_at_bats = model.fit()

# Print the results

print(model_fitted_at_bats.summary())

We can read the intercept and slope values from the results, or we can print the intercept and slope values as follows.

In [ ]:
print('Intercept =', model_fitted_at_bats.params[0])
print('Slope =', model_fitted_at_bats.params[1])

Knowing the intercept and slope, we can write down the least squares regression line for the linear model as follows.

> $y = - 2789.2429 + 0.6305 \times \text{at_bats}$

One last piece of information we will discuss from the summary output is the **Multiple R-squared**, or more simply, ${R}$<sup>2</sup>. The ${R}$<sup>2</sup> value represents the proportion of variability in the response variable that is explained by the explanatory variable.

For this model, 37.3% of the variability in `runs` is explained by `at_bats`. This can be read as the `R-squared` value in the summary printout above, or printed directly with the code below.

In [ ]:
print('R-squared =', model_fitted_at_bats.rsquared)

### Exercise 3

- Fit a new model that uses <code>homeruns</code> to predict <code>runs</code>.
- Using the estimates from the Python output, write the equation of the regression line.
- What percent of the variability in `runs` is explained by `homeruns`?
- What does the slope tell us in the context of the relationship between success of a team and its home runs?

In [ ]:
# Use `statsmodel` as above to predict `runs` with `homeruns`

# Set the formula string (which variables to compare, "reponse ~ predictor")
# and the dataframe to use.

formula_string = "???" # Fill in the formula string
dataframe = mlb11

# Fit the linear model

model = sm.formula.ols(formula = formula_string, data = dataframe)
model_fitted_homeruns = model.fit()

# Print the results

print(model_fitted_homeruns.summary())

## Prediction and prediction errors

Just as we used the mean and standard deviation to summarize a single variable, we can summarize the relationship between these two variables by finding the line that best follows their association. Let's plot `at_bats` and `runs` on a scatter plot along with its least squares regression line.

In [ ]:
(
    ggplot(mlb11) +
    aes(x='at_bats', y='runs') +
    geom_point() +
    geom_smooth(method = 'lm', se = False)
)

### Exercise 4

- If a team manager saw the least squares regression line and not the actual data, how many runs would he or she predict for a team with 5,710 at-bats?
- Is this an overestimate or an underestimate, and by how much? In other words, what is the residual for this prediction?

## Model diagnostics

To assess whether the linear model is reliable, we need to check for
1. linearity,
2. nearly normal residuals, and
3. constant variability.

**Linearity.** You already checked if the relationship between runs and at-bats is linear using a scatterplot. We should also verify this condition with a plot of the residuals vs. at-bats.

In [ ]:
import seaborn as sns

sns.residplot(x='at_bats', y='runs', data=mlb11, color='red')

plt.xlabel('at_bats', fontsize = 12)
plt.ylabel('residuals', fontsize = 12)
plt.show();

### Exercise 5

- Is there any apparent pattern in the residuals plot?
- What does this indicate about the linearity of the relationship between runs and at-bats?

**Nearly normal residuals.** To check this condition, we can look at a histogram.

In [ ]:
residuals = mlb11['runs'] - model_fitted_at_bats.predict(mlb11['at_bats'])

(
    ggplot(pd.DataFrame({'residuals': residuals})) +
    aes(x = 'residuals') +
    geom_histogram()
)

We should also look at a normal probability plot (i.e., Q-Q plot) of the residuals.

In [ ]:
def qq_plot(dataframe, variable, title):
  plot = (
      ggplot(dataframe) +
      aes(sample = variable) +
      stat_qq() +
      stat_qq_line() +
      labs(title = title)
  )
  return plot

qq_plot(pd.DataFrame({'residuals': residuals}), 'residuals', 'Q-Q plot of residuals')

### Exercise 6

Based on the histogram and the normal probability plot, does the nearly normal residuals condition appear to be met?

**Constant variability.** The variation of points around the regression line should be approximately constant.

### Exercise 7

Based on the plot above Exercise 4 (showing the data points along with the regression line), does the constant variability condition appear to be met?

## Examining other variables

Python can print a variety of correlation coefficients simultaneously with the `.corr()` command. The following code produces the correlation coefficients for each pair of the traditional variables.

In [ ]:
mlb11[['runs', 'at_bats', 'homeruns', 'bat_avg', 'strikeouts', 'stolen_bases', 'wins']].corr()

This table shows us the correlation coefficient for `runs` and `at_bats` to be 0.610627, as computed earlier. Meanwhile, the correlation coefficient for `strikeouts` and `runs` is -0.411531.

Note, the traditional variable with strongest correlation to `runs` is `bat_avg`, with a correlation coefficient of 0.809986.

### Exercise 8

- Use `.corr()` to create a table of correlation coefficients for `runs` as compared to the three new variables `new_onbase`, `new_slug`, and `new_obs`.
- Which of these new variables has the strongest correlation to `runs`?
- Is this correlation stronger than those found for the traditaional variables?

In [ ]:
# Use .corr() to find the correlation coefficients for `runs` as compared to the three new
# variables `new_onbase`, `new_slug`, and `new_obs`



### Exercise 9

- Identify which variable (traditional or new) you expect to be the best predictor for `runs`.
- Make a scatterplot with this variable as a predictor for `runs`.
- Describe the relationship between these two variables. Make sure to discuss the form, direction, and strength of the relationship as well as any unusual observations.

In [ ]:
# Make a scatterplot



### Exercise 10

Continuing from Exercise 9,
- Determine an equation for the regression line for your chosen predictor variable and `runs`.
- What proportion of the variability in `runs` is explained by your predictor variable?

```
Hint: Use a printout from `statsmodel`, similar to Exercise 3.
```

In [ ]:
# Use `statsmodel` to print a summary

# Set the formula string (which variables to compare, "reponse ~ predictor")
# and the dataframe to use.

formula_string = "???" # Fill in the formula string!
dataframe = mlb11

# Fit the linear model

model = sm.formula.ols(formula = formula_string, data = dataframe)
model_fitted = model.fit()

# Print the results

print(model_fitted.summary())

### Exercise 11

Continuing from Exercise 10,
- Make a scatter plot of your variables with their regression line.
- Use `.residplot()` to plot the residuals.
- Then, use `qq_plot()` to make a Q-Q plot for the residuals.
- Use these plots to determine whether the **linearity**, **nearly normal residuals**, and **constant variance** conditions appear to be met.

```
Hint: See the 'Model diagnostics' section for guidance!
```

In [ ]:
# Plot the data along with the regression line
# See Exercise 4 for reference!



In [ ]:
# Fill in the code to use .residplot() to plot the residuals
# See Exercise 5 for reference!

sns.residplot(x='???', y='???', data=mlb11, color='red')

plt.xlabel('new_obs', fontsize = 12)
plt.ylabel('residuals', fontsize = 12)
plt.show();

In [ ]:
# Fill in the code to use qq_plot() to make a Q-Q plot for the residuals to assess the normality of
# the residuals. See Exercise 6 for reference!

residuals = mlb11['runs'] - model_fitted.predict(mlb11['???'])

qq_plot(pd.DataFrame({'residuals': residuals}), 'residuals', 'Q-Q plot of residuals')

### Exercise 12

Continuing from Exercise 11,
- Does the data provide convincing evidence to conclude that the variable you've chosen is actually correlated to `at_runs`? Make your conclusion using a p-value and significance level $\alpha = 0.05$.

```
Hint: Refer to the summary printout from `statsmodel` in Exercise 10.
```

---

This lab was adapted by Timothy L. Clark, derivative of [OpenIntro Statistics by Diez, Çetinkaya-Rundel, and Barr](https://www.openintro.org/book/os/), released under [Creative Commons BY-SA 3.0](https://creativecommons.org/licenses/by-sa/3.0/deed.en) license.